# Corrected LSTM — Fair Evaluation / 修正版 LSTM——公平评估

The original LSTM (`01_lstm_reimplementation.ipynb`) achieved 99.1% accuracy, but EDA revealed two leakage sources:
- `subject` column: perfectly separates fake from real (100% accuracy alone)
- Reuters byline in text: present in 99.2% of real news, 0% of fake news

This notebook retrains the same LSTM architecture with **clean input only**: `title + text`, with `subject` and `date` removed. The goal is to measure how well the model performs when it cannot exploit source-level shortcuts — and to provide a fair baseline for comparison with DistilBERT.

---

原始 LSTM（`01_lstm_reimplementation.ipynb`）达到了 99.1% 的准确率，但 EDA 发现了两个泄露来源：
- `subject` 列：单独使用就能 100% 区分真假新闻
- 正文中的 Reuters 署名：出现在 99.2% 的真新闻中，假新闻中为 0%

本 notebook 使用完全相同的 LSTM 结构重新训练，但输入改为**干净的 `title + text`**，去掉 `subject` 和 `date`。目标是在模型无法利用来源捷径的情况下，测量其真实的语言理解能力，并为后续与 DistilBERT 的比较提供公平的基准。

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

os.makedirs('../results', exist_ok=True)
print('TensorFlow:', tf.__version__)

TensorFlow: 2.16.2


## 2. Load Data — Clean Input Only

与原始 LSTM 的关键区别：输入改为 `title + text` 拼接，去掉 `subject` 和 `date` 列，消除泄露来源。

Key difference from the original LSTM: input is now `title + text` concatenated. `subject` and `date` are excluded to remove leakage sources.

In [ ]:
import html as html_module

fake = pd.read_csv('../data/raw/Fake.csv')
true = pd.read_csv('../data/raw/True.csv')

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true]).sample(frac=1, random_state=42).reset_index(drop=True)

# Fix 1: Remove Reuters byline at start, handles multi-word city names
# 去掉开头的 Reuters 署名，支持多词城市名
df['text'] = df['text'].str.replace(r'[\w\s]+\(Reuters\)\s*-?\s*', '', regex=True)

# Fix 2: Remove remaining Reuters mentions anywhere in the body
# 去掉正文中其他地方出现的 Reuters 字样
df['text'] = df['text'].str.replace(r'Reuters', '', regex=False)

# Fix 3: Remove residual all-caps city datelines at start of text
# 去掉残留的开头全大写城市署名
df['text'] = df['text'].str.replace(r'^\s*[A-Z][A-Z\s\.]+\s*-\s*', '', regex=True)

# Fix 4: Remove other wire service bylines (AP, AFP, UPI)
# 去掉其他通讯社署名
df['text'] = df['text'].str.replace(r'[\w\s]+\((AP|AFP|UPI)\)\s*-?\s*', '', regex=True)
df['text'] = df['text'].str.replace(r'\b(AP|AFP|UPI)\b', '', regex=True)

# Fix 5: Remove URLs
# 去掉 URL 链接
df['text'] = df['text'].str.replace(r'http\S+|www\.\S+', '', regex=True)

# Fix 6: Remove social media handles
# 去掉社交媒体 @用户名
df['text'] = df['text'].str.replace(r'@\w+', '', regex=True)

# Fix 7: Unescape HTML entities and remove non-ASCII characters
# 解码 HTML 实体字符，去掉非 ASCII 字符
df['text'] = df['text'].apply(html_module.unescape)
df['text'] = df['text'].str.replace(r'[^\x00-\x7F]+', ' ', regex=True)

# Fix 8: Normalise whitespace
# 统一处理多余空白
df['text'] = df['text'].str.strip()
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)

# 拼接 title 和 text，去掉 subject 和 date
# Concatenate title and text — subject and date excluded
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Fix 9: Remove articles that became too short after cleaning (< 10 words)
# 去掉清洗后内容过短的文章（少于 10 词）
before = len(df)
df = df[df['content'].str.split().str.len() > 10].reset_index(drop=True)
after = len(df)

print(f'Total after cleaning: {after:,} rows ({before - after} removed as too short)')
print('Cleaning applied:')
print('  [1-3] Reuters byline, mentions, city datelines removed')
print('  [4]   AP / AFP / UPI bylines removed')
print('  [5]   URLs removed')
print('  [6]   Social media handles removed')
print('  [7]   HTML entities decoded, non-ASCII removed')
print('  [8]   Whitespace normalised')
print('  [9]   Articles with fewer than 10 words removed')
df[['content', 'label']].head(3)

共九步清洗，涵盖所有可识别的来源层面泄露信号。最后一步过滤掉清洗后内容过短的文章——这些文章在大量去除文本后已经没有足够内容让模型学习，保留反而会引入噪声。

唯一无法消除的剩余偏差是 Reuters 的**写作风格**（正式语气、倒金字塔结构），将在 error analysis 中作为局限性提出。

Nine cleaning steps applied, covering all identifiable source-level leakage signals. The final step removes articles that became too short after cleaning — these contain insufficient content for the model to learn from and would introduce noise. The only remaining bias is Reuters' **writing style**, noted as a limitation in error analysis.

## 3. Text Preprocessing / 文本预处理

按照原论文的流程，文本预处理分三步：划分训练测试集 → Tokenizer 建立词表 → 将文本转换为数字序列并统一长度（padding）。输入使用 `content`（title + text），不含 `subject` 和 `date`。

Following the original paper pipeline: train/test split → Tokenizer vocabulary construction → convert text to padded integer sequences. Input is `content` (title + text only).

In [5]:
# 划分训练测试集 80/20，与原论文一致
# Train/test split 80/20, matching original paper
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

Train: 35,918  |  Test: 8,980


In [6]:
# Tokenizer：保留前 5000 高频词，与原论文一致
# Tokenizer: top 5,000 words, matching original paper
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index)
print(f'Full vocabulary size: {vocab_size:,}')
print(f'Using top 5,000 words (as per original paper)')

Full vocabulary size: 113,246
Using top 5,000 words (as per original paper)


In [7]:
# 转为数字序列并补齐至长度 200，与原论文一致
# Convert to sequences and pad to length 200, matching original paper
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=200)
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=200)

print(f'X_train shape: {X_train_seq.shape}')
print(f'X_test shape:  {X_test_seq.shape}')

X_train shape: (35918, 200)
X_test shape:  (8980, 200)


预处理完成。每篇文章被表示为长度 200 的整数数组，不足补零，超过截断。与原始 LSTM 的唯一区别是输入内容：这里是 `title + text`，原始版本是 `text`。

Preprocessing complete. Each article is a zero-padded integer sequence of length 200. The only difference from the original LSTM is the input: here we use `title + text`; the original used `text` only.